## <b>Movie Recommendation System</b>

### <b>Step 3 : Data Cleaning & Feature Engineering</b>

#### <b>Objective:</b>

The objective of this phase is to prepare the movie dataset for building a content-based recommendation system. Raw movie metadata contains missing values, inconsistent text formats, and multiple features spread across different columns. These issues must be addressed before applying machine learning techniques.

##### <b>Data Cleaning</b>

Several preprocessing steps were performed to improve data quality:

- Selected only the features relevant for recommendations:
    - IMDb ID
    - Title
    - Genre
    - Director
    - Writer
    - Actors
    - Plot
    - IMDb Rating
- Handled missing values:
    - Missing text fields were replaced with empty strings.
    - Missing IMDb ratings were filled using the median rating of the dataset.
- Converted IMDb ratings to numeric format for future ranking and evaluation purposes.

##### <b>Feature Engineering</b>

To represent each movie using its content information, multiple textual attributes were combined into a single feature called tags.

The following columns were used:
- Genre
- Director
- Writer
- Actors
- Plot

The preprocessing steps included:
1. Splitting comma-separated values into lists.
2. Removing spaces from names such as:
    - "Christopher Nolan" → "ChristopherNolan"
    - "Leonardo DiCaprio" → "LeonardoDiCaprio"
3. Tokenizing movie plots into individual words.
4. Combining all features into a unified text representation.
5. Converting all text to lowercase to ensure consistency.

In [1]:
# Import Libraries
import pandas as pd
import numpy as np

In [2]:
# Load the Dataset
df = pd.read_csv('../data/Netflix_data.csv')
print("Dataset Shape:", df.shape)

Dataset Shape: (3516, 17)


In [3]:
# Select Only Relevant Columns
movies = df[['imdbID', 'Title', 'Genre', 'Director', 'Writer', 'Actors', 'Plot', 'imdbRating']].copy()
movies.head()

,imdbID,Title,Genre,Director,Writer,Actors,Plot,imdbRating
0,tt0107362,Last Action Hero,"Action, Adventure, Comedy",John McTiernan,"Zak Penn, Adam Leff, Shane Black","Arnold Schwarzenegger, F. Murray Abraham, Art ...",Young Danny Madigan is a huge fan of Jack Slat...,6.5
1,tt21191806,Back in Action,"Action, Comedy",Seth Gordon,"Seth Gordon, Brendan O'Brien, Lillian Yu","Jamie Foxx, Cameron Diaz, McKenna Roberts",Former CIA spies Emily and Matt are pulled bac...,5.9
2,tt0318155,Looney Tunes: Back in Action,"Animation, Adventure, Comedy",Joe Dante,Larry Doyle,"Brendan Fraser, Jenna Elfman, Steve Martin",Bugs Bunny and Daffy Duck are up to their feud...,5.8
3,tt15600222,An Action Hero,"Action, Comedy, Crime",Anirudh Iyer,"Anirudh Iyer, Neeraj Yadav","Ayushmann Khurrana, Jaideep Ahlawat, Gautam Jo...",A murder accusation turns a movie star's own l...,7.0
4,tt0120633,A Civil Action,"Biography, Drama",Steven Zaillian,"Jonathan Harr, Steven Zaillian","John Travolta, Robert Duvall, Tony Shalhoub","Jan Schlichtmann, a tenacious lawyer, is addre...",6.6


In [4]:
# Check Missing Values
print("Missing Values:\n", movies.isnull().sum())

Missing Values:
 imdbID          0
Title           0
Genre          40
Director      961
Writer        974
Actors        228
Plot          357
imdbRating    456
dtype: int64


In [5]:
# Fill Missing Values
movies['Genre'] = movies['Genre'].fillna("")
movies['Director'] = movies['Director'].fillna("")
movies['Writer'] = movies['Writer'].fillna("")
movies['Actors'] = movies['Actors'].fillna("")
movies['Plot'] = movies['Plot'].fillna("")

In [6]:
movies.isnull().sum()

imdbID          0
Title           0
Genre           0
Director        0
Writer          0
Actors          0
Plot            0
imdbRating    456
dtype: int64

In [7]:
# Clean Imdb Ratings
movies['imdbRating'] = pd.to_numeric(movies['imdbRating'], errors='coerce')
movies['imdbRating'] = movies['imdbRating'].fillna(movies['imdbRating'].median())

In [8]:
movies.isnull().sum()

imdbID        0
Title         0
Genre         0
Director      0
Writer        0
Actors        0
Plot          0
imdbRating    0
dtype: int64

In [ ]:
# Convert Genres to List
movies['Genre'] = movies['Genre'].apply(lambda x : x.split(', '))

In [10]:
# Convert Directors to List
movies['Director'] = movies['Director'].apply(lambda x : x.split(', '))

In [11]:
# Convert Writer to List
movies['Writer'] = movies['Writer'].apply(lambda x : x.split(', '))

In [12]:
# Convert Actors to List
movies['Actors'] = movies['Actors'].apply(lambda x : x.split(', '))

In [13]:
# Convert Plot into Words
movies['Plot'] = movies['Plot'].apply(lambda x : x.split())

In [14]:
# Remove Spaces from Genres, Directors, Writers, Actors
movies['Genre'] = movies['Genre'].apply(lambda x : [i.replace(" ", "") for i in x])
movies['Director'] = movies['Director'].apply(lambda x : [i.replace(" ", "") for i in x])
movies['Writer'] = movies['Writer'].apply(lambda x : [i.replace(" ", "") for i in x])
movies['Actors'] = movies['Actors'].apply(lambda x : [i.replace(" ", "") for i in x])



In [15]:
# Create Tags Column
movies['tags'] = movies['Genre'] + movies['Director'] + movies['Writer'] + movies['Actors'] + movies['Plot']


In [16]:
# Convert tags list to text
movies['tags'] = movies['tags'].apply(lambda x : " ".join(x))

In [17]:
# Convert tags to lowercase
movies['tags'] = movies['tags'].str.lower()

In [18]:
# Create Final Dataset
final_df = movies[['imdbID', 'Title', 'imdbRating', 'tags']].copy()

print("Final Dataset Shape:", final_df.shape)
final_df.head()

Final Dataset Shape: (3516, 4)


,imdbID,Title,imdbRating,tags
0,tt0107362,Last Action Hero,6.5,action adventure comedy johnmctiernan zakpenn ...
1,tt21191806,Back in Action,5.9,action comedy sethgordon sethgordon brendano'b...
2,tt0318155,Looney Tunes: Back in Action,5.8,animation adventure comedy joedante larrydoyle...
3,tt15600222,An Action Hero,7.0,action comedy crime anirudhiyer anirudhiyer ne...
4,tt0120633,A Civil Action,6.6,biography drama stevenzaillian jonathanharr st...


In [ ]:
# Save the Dataset
final_df.to_csv('../data/Netflix_Dataset.csv', index=False)

print("Feature Engineering Completed and Dataset Saved!")

Feature Engineering Completed and Dataset Saved!


##### <b>Why Create a Tags Column?</b>

Machine learning algorithms cannot directly understand multiple text columns independently. That's why combining relevant movie information into a single textual feature will allows us to transform movies into numerical vectors using Natural Language Processing (NLP) techniques such as TF-IDF.

##### <b>Outcome</b>

##### <b>Final Dataset</b>

After preprocessing, the dataset contains the following columns:

|Column|	Description|
|----------|-----------|
|imdbID|	Unique IMDb identifier|
|Title	|Movie or TV Series title|
|imdbRating	|IMDb user rating|
|tags	|Combined textual representation of movie content|

The resulting dataset is clean, structured, and ready for vectorization. In the next phase, the tags column will be transformed into numerical vectors using TF-IDF, and cosine similarity will be used to identify movies with similar content, forming the foundation of the recommendation engine.